# ReAct: Reasoning and Acting

**Prerequisites:** Agent loops and tool-use basics. **Estimated time:** 35 minutes.

**Objectives:** Separate a proposed tool action from its observation; validate tool inputs; stop safely on errors or a step limit.

## Concept recap and assumptions

A ReAct loop alternates a decision with an observation. Here the policy is an ordinary deterministic function over a local inventory tool. The trace is the useful artifact: it shows what was requested, what the tool returned, and why the loop stopped.

## 1. Transparent local baseline

Run this deterministic, dependency-free baseline. All records are synthetic, and the code has no network access or side effects.

In [ ]:
import random
random.seed(20260922)
inventory = {'blue mug': 4, 'green mug': 0}

def lookup_stock(item):
    if item not in inventory: return {'ok': False, 'error': 'unknown_item'}
    return {'ok': True, 'count': inventory[item]}

def run(query, max_steps=3):
    trace = []
    for _ in range(max_steps):
        # Small policy: parse a known item from the query, then call read-only lookup.
        item = next((name for name in inventory if name in query), None)
        if item is None: return trace, 'clarify'
        observation = lookup_stock(item)
        trace.append({'action': 'lookup_stock', 'item': item, 'observation': observation})
        if not observation['ok']: return trace, 'tool_error'
        return trace, ('available' if observation['count'] > 0 else 'out_of_stock')
    return trace, 'step_limit'

trace, answer = run('Is the blue mug in stock?')
assert answer == 'available' and trace[0]['observation']['count'] == 4
print(trace, answer)

## 2. Extension / realistic constraint

Add a schema check for the tool argument and an explicit `clarify` response for ambiguous item names. Keep the tool read-only; this example does not model a language model or prove tool-call correctness.

## 3. Comparison and evaluation

Track task success, invalid tool arguments, tool errors, and number of steps separately. A correct final answer with an invalid intermediate call can still violate a production contract.

## 4. Edge and failure cases

Check an unknown product and an empty query. Add a tool that raises an exception and convert it to a typed error observation; never let an untrusted tool result become executable code.

## 5. Exercises

Add a quantity tool that accepts only a known item and integer request. Tests: unknown item is rejected; zero stock never yields an availability claim; each run makes at most one lookup. State what the policy cannot infer.

## Summary and next practice

ReAct makes observations explicit and debuggable. A bounded loop, validated arguments, and typed tool errors prevent retries from becoming an uncontrolled action sequence.

Related practice: agentic-ai/interview-prep/agentic-ai-interview-questions.md — tool loops, observability, and safe stopping.